# Visualization of FWI Results & Comparison

In [1]:
import re
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.animation import FuncAnimation, writers
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from cartopy.io.img_tiles import GoogleTiles
from IPython.display import HTML


# Set up formatting for the movie files
Writer = writers['ffmpeg']
writer = Writer(fps=1, metadata=dict(artist='Me'), bitrate=1800)

# fwi: a 2D DataArray, e.g. ds["fwi"].isel(time=0)
# Example display boundaries below 30; adjust to your chosen classification.
bounds = [0, 5, 10, 20, 30]

cmap = ListedColormap(["#2c7bb6", "#abd9e9", "#fdae61", "#d7191c"])
cmap.set_over("#67001f")  # FWI >= 30: extreme
#cmap.set_bad("#d9d9d9")   # Missing data

norm = BoundaryNorm(bounds, ncolors=cmap.N, clip=False)

tiler = GoogleTiles(style='satellite')
mercator = tiler.crs


In [2]:
def group_granules(input_dir: Path) -> list[Path]:
    for path in input_dir.rglob(pattern='*.csv'):
        file : Path = path.resolve()
    return file

FIRM_DATA_DIR = Path('../../../firms_modis_mcd14/DL_FIRE_M-C61_811179/')
grouped_mcd61 = group_granules(FIRM_DATA_DIR)

df = pd.read_csv(grouped_mcd61, parse_dates=['acq_date'])
df["acq_date"] = df["acq_date"].dt.normalize()
df = df[df["type"]==0].copy()
fire_gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)
fire_gdf.tail(3)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,geometry
6031,5.7524,-72.6674,309.9,1.7,1.3,2023-01-31,1812,Aqua,MODIS,50,61.03,294.2,16.8,D,0,POINT (-72.6674 5.7524)
6032,5.8528,-72.3146,319.4,1.6,1.2,2023-01-31,1812,Aqua,MODIS,55,61.03,300.0,26.5,D,0,POINT (-72.3146 5.8528)
6033,5.8509,-72.3271,311.9,1.6,1.2,2023-01-31,1812,Aqua,MODIS,27,61.03,295.0,16.5,D,0,POINT (-72.3271 5.8509)


In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(
    [
        -84, -45, 
        -25, 10
    ], 
    crs=ccrs.PlateCarree()
)
ax.add_image(tiler, 6)
title = ax.set_title('')


OUTPUT_DIR = Path('/Users/kris/Documents/grad/fwi_january_2023/')
OUTPUT_FILE = sorted(
    file.resolve() for file in OUTPUT_DIR.rglob(pattern='fwi_*.nc')
    if re.fullmatch(r"fwi_\d{8}\.nc", file.name)
)
ds = xr.open_mfdataset(OUTPUT_FILE)
da_fwi = ds['fwi']

im = da_fwi.isel(time=0).plot(
    ax=ax, 
    transform=ccrs.PlateCarree(),
    x='east_west', 
    y='north_south', 
    norm=norm,
    animated=True,
    extend='max', cmap=cmap,
    zorder=2,
    cbar_kwargs={'label': 'Fire Weather Index'}
)
points = ax.scatter(
    [],
    [],
    s=20,
    facecolor='None',
    edgecolors='black',
    transform = ccrs.PlateCarree(),
    marker="o",
    zorder=20,
    label = "Active Fire Event"
)


def update(frame):
    im.set_array(da_fwi.isel(time=frame).values.ravel())
    current_time = da_fwi.time.values[frame].astype("datetime64[D]")
    current_fire = fire_gdf[
        fire_gdf['acq_date'] == current_time
    ]
    xy = np.column_stack([
        current_fire.geometry.x.to_numpy(),
        current_fire.geometry.y.to_numpy()
    ])
    points.set_offsets(xy)
    title.set_text(f"LIS/LDAS - Hindcast {da_fwi.time.values[frame].astype("datetime64[D]")} FWI — daily-mean pilot")
    return im, points, title
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAKES, alpha=0.9)
ax.add_feature(cfeature.RIVERS)

ani = FuncAnimation(
    fig,
    update,
    frames=da_fwi.sizes['time'],
    interval=500,
    blit=False,
    
)
#fig.tight_layout()
plt.close(fig)
ani.save('../../../outputs/lis_ldas_fwi_active_fire.mp4', writer=writer)
# HTML(ani.to_jshtml())

In [ ]:
# ax.set_title('January 1, 2023 FFMC — daily-mean pilot, initial FFMC 85')
# ax.set_xlabel('LDAS east_west grid index')
# ax.set_ylabel('LDAS north_south grid index')
# fig.tight_layout()

#fig.savefig(OUTPUT / 'ffmc_20230101.png', dpi=150)
#plt.close(fig)

# from IPython.display import Image, display
# display(Image(filename=str(OUTPUT / 'ffmc_20230101.png')))

## GSFC GEOS-5 FWI

In [ ]:
# %%bash
# cd /mnt/e/backup/GSFC_GFWED_FWI/
# for DAY in $(seq -w 1 31); do
#     wget -c "https://portal.nccs.nasa.gov/datashare/GlobalFWI/v2.0/fwiCalcs.GEOS-5/Default/GEOS-5/2023/FWI.GEOS-5.Daily.Default.202301${DAY}.nc"
# done

--2026-09-23 21:14:03--  https://portal.nccs.nasa.gov/datashare/GlobalFWI/v2.0/fwiCalcs.GEOS-5/Default/GEOS-5/2023/FWI.GEOS-5.Daily.Default.20230101.nc
Resolving portal.nccs.nasa.gov (portal.nccs.nasa.gov)... 169.154.151.145, 2001:4d0:2418:2800::a99a:9791
Connecting to portal.nccs.nasa.gov (portal.nccs.nasa.gov)|169.154.151.145|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17210240 (16M) [application/octet-stream]
Saving to: ‘FWI.GEOS-5.Daily.Default.20230101.nc’

     0K .......... .......... .......... .......... ..........  0% 6.33M 3s
    50K .......... .......... .......... .......... ..........  0% 1.34M 7s
   100K .......... .......... .......... .......... ..........  0% 2.72M 7s
   150K .......... .......... .......... .......... ..........  1% 3.10M 6s
   200K .......... .......... .......... .......... ..........  1% 4.52M 6s
   250K .......... .......... .......... .......... ..........  1% 4.92M 5s
   300K .......... .......... .......... ......

In [3]:
def preprocess(ds):
    filename = Path(ds.encoding["source"])

    # Extract 20230101 from filename
    date_str = filename.stem.split(".")[-1]
    date = pd.to_datetime(date_str, format="%Y%m%d")

    # Existing time dimension has length 1;
    # replace its index with the actual date
    ds = ds.assign_coords(time=[date])

    return ds

In [4]:
BBOX = (
    -82.0, -21.0, 
    -49.0, 6.0
)

GSFC_GFWED_FWI = Path('/mnt/e/backup/GSFC_GFWED_FWI/2023')

GSFC_GFWED_FWI.exists()

GFWED_202301_FWI_FILE = sorted([file for file in GSFC_GFWED_FWI.glob(pattern='*.nc')])
gfwed_202301_ds = xr.open_mfdataset(
    GFWED_202301_FWI_FILE, 
    preprocess=preprocess, 
    engine='netcdf4'
)
# gfwed_202301_ds.attrs.update({'History' : '2023 Januaray, daily fwi data'})
gfwed_202301_da_fwi = gfwed_202301_ds['GEOS-5_FWI']
gfwed_202301_ds.close()
gfwed_202301_da_fwi = gfwed_202301_da_fwi.sel(
    lat = slice(-21.0, 6.0), 
    lon = slice(-82.0, -49.0),
)

In [5]:
fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(
    [
        -84, -45, 
        -25, 10
    ], 
    crs=ccrs.PlateCarree()
)
ax.add_image(tiler, 6)
title = ax.set_title('')


im = gfwed_202301_da_fwi.isel(time=0).plot(
    ax=ax, 
    transform=ccrs.PlateCarree(),
    x='lon', 
    y='lat', 
    norm=norm,
    animated=True,
    extend='max', cmap=cmap,
    zorder=2,
    cbar_kwargs={'label': 'Fire Weather Index'}
)
points = ax.scatter(
    [],
    [],
    s=20,
    facecolor='None',
    edgecolors='black',
    transform = ccrs.PlateCarree(),
    marker="o",
    zorder=20,
    label = "Active Fire Event"
)


def update(frame):
    im.set_array(gfwed_202301_da_fwi.isel(time=frame).values.ravel())
    current_time = gfwed_202301_da_fwi.time.values[frame].astype("datetime64[D]")
    current_fire = fire_gdf[
        fire_gdf['acq_date'] == current_time
    ]
    xy = np.column_stack([
        current_fire.geometry.x.to_numpy(),
        current_fire.geometry.y.to_numpy()
    ])
    points.set_offsets(xy)
    title.set_text(f"NASA GSFC GFWED {gfwed_202301_da_fwi.time.values[frame].astype("datetime64[D]")} GEOS-5 FWI — daily pilot")
    return im, points, title
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAKES, alpha=0.9)
ax.add_feature(cfeature.RIVERS)

ani = FuncAnimation(
    fig,
    update,
    frames=gfwed_202301_da_fwi.sizes['time'],
    interval=500,
    blit=False,
    
)
#fig.tight_layout()
plt.close(fig)
ani.save('../../../outputs/gfwed_202301_fwi_active_fire.mp4', writer=writer)
# HTML(ani.to_jshtml())